# Level 2 — Portfolio Construction

**Audience:** users comfortable with return statistics who want to construct
long-only portfolios.

**Prerequisites:** Level 1 or equivalent knowledge of annualized return and
volatility.

**Learning goals**

1. estimate annual expected returns and covariance from periodic data;
2. evaluate a weight vector;
3. calculate minimum-volatility, maximum-Sharpe, and GMV portfolios;
4. generate and inspect an efficient frontier.

This notebook distills the reusable ideas from legacy labs 107–111 and 118.

## 1. Setup and synthetic asset returns

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

pd.options.display.float_format = "{:.4f}".format

from asset_management_toolkit.portfolio import (
    efficient_frontier_weights,
    global_minimum_variance,
    maximum_sharpe_ratio,
    minimum_volatility,
    portfolio_return,
    portfolio_volatility,
)

In [ ]:
rng = np.random.default_rng(42)
monthly_mean = np.array([0.0040, 0.0065, 0.0080, 0.0030])
monthly_cov = np.array(
    [
        [0.00040, 0.00010, 0.00008, 0.00003],
        [0.00010, 0.00090, 0.00025, 0.00004],
        [0.00008, 0.00025, 0.00160, 0.00002],
        [0.00003, 0.00004, 0.00002, 0.00016],
    ]
)
asset_names = ["Bonds", "Quality", "Equity", "Defensive"]
sample = pd.DataFrame(
    rng.multivariate_normal(monthly_mean, monthly_cov, size=120),
    columns=asset_names,
)
expected_returns = sample.mean() * 12
covariance = sample.cov() * 12
expected_returns

## 2. Evaluate an existing allocation

Weights and expected returns must have the same order. Labeled pandas inputs
make that contract visible.

In [ ]:
equal_weight = pd.Series(0.25, index=asset_names)
pd.Series(
    {
        "expected_return": portfolio_return(equal_weight, expected_returns),
        "volatility": portfolio_volatility(equal_weight, covariance),
    },
    name="equal_weight",
)

## 3. Compare standard long-only portfolios

In [ ]:
target = float(expected_returns.median())
weights = pd.DataFrame(
    {
        "Min vol at target": minimum_volatility(
            target, expected_returns, covariance
        ),
        "Max Sharpe": maximum_sharpe_ratio(
            0.02, expected_returns, covariance
        ),
        "GMV": global_minimum_variance(covariance),
        "Equal weight": equal_weight,
    }
)
weights

In [ ]:
portfolio_table = pd.DataFrame(
    {
        name: {
            "expected_return": portfolio_return(w, expected_returns),
            "volatility": portfolio_volatility(w, covariance),
        }
        for name, w in weights.items()
    }
).T
portfolio_table["ex_ante_sharpe"] = (
    portfolio_table["expected_return"] - 0.02
) / portfolio_table["volatility"]
portfolio_table

## 4. Efficient frontier

Each row below is the minimum-volatility long-only allocation for a target
return. The index stores the target return so the result remains auditable.

In [ ]:
frontier_weights = efficient_frontier_weights(
    15, expected_returns, covariance
)
frontier = pd.DataFrame(
    {
        "expected_return": [
            portfolio_return(row, expected_returns)
            for _, row in frontier_weights.iterrows()
        ],
        "volatility": [
            portfolio_volatility(row, covariance)
            for _, row in frontier_weights.iterrows()
        ],
    },
    index=frontier_weights.index,
)
frontier.head()

## Exercise

Find the minimum-volatility portfolio targeting 7% annual return. Verify that
weights sum to one and that the realized model return matches the target.

In [ ]:
exercise_weights = minimum_volatility(
    0.07, expected_returns, covariance
)
# Add your verification below.

### Answer scaffold

In [ ]:
pd.Series(
    {
        "weight_sum": exercise_weights.sum(),
        "minimum_weight": exercise_weights.min(),
        "model_return": portfolio_return(
            exercise_weights, expected_returns
        ),
        "model_volatility": portfolio_volatility(
            exercise_weights, covariance
        ),
    }
)

## Common pitfalls and next steps

- Expected returns and covariance must use the same annualization convention.
- These optimizers are long-only and fully invested.
- Estimated expected returns are noisy; compare Max Sharpe with GMV and equal
  weight rather than treating one optimizer as truth.
- Optimization outputs are model allocations, not investment advice.

Next: Level 3 combines analytics, benchmark-relative diagnostics, and an
in-sample/out-of-sample review.